In [1]:
import import_ipynb
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import pickle
from datetime import datetime
from collections import defaultdict

from functions import *
from params import *

# Exact Trotter evolution

In [ ]:
epsilon = 0.05
ferm_num_exact_no_noise_no_knit = np.zeros(0)
bs_error_exact_no_noise_no_knit = np.zeros(0)
for i in range(32):
    print(i)
    step = int(4*(i+1))
    num_shots = 1024**2
    circuit = trotter_stepper(step, Nqbits, epsilon, mass, mid).decompose().decompose()
    circuit.measure_all()
    res = do_run(circuit, num_shots)
    ferm_num_exact_no_noise_no_knit=np.append(ferm_num_exact_no_noise_no_knit, fermion_number(res, mid))
    bs_error_exact_no_noise_no_knit=np.append(bs_error_exact_no_noise_no_knit,bootstrap_error(res, mid, num_shots))

In [ ]:
ferm_num_exact_no_noise_no_knit = np.array(ferm_num_exact_no_noise_no_knit)
bs_error_exact_no_noise_no_knit = np.array(bs_error_exact_no_noise_no_knit)
trotter_steps = np.array(trotter_steps)

In [ ]:
np.save('data/ferm_num_exact_no_noise_no_knit', ferm_num_exact_no_noise_no_knit)
np.save('data/bs_error_exact_no_noise_no_knit', bs_error_exact_no_noise_no_knit)

# noisy, no knit Trotter evolution

## do runs

In [2]:
def make_runs(my_seed=None, shots=2*8192):
    if not my_seed:
        my_seed = int(datetime.now().timestamp())
    ferm_num_trotter_evol_noisy_no_knit = np.empty(0)
    bs_error_trotter_evol_noisy_no_knit = np.empty(0)
    for eps in np.linspace(0, 1.6, 9):
        internal_seed = my_seed
        eps=np.round(eps, 1)
        print(eps)
        np.random.seed(internal_seed)
        my_seed=np.random.randint(1024**2)
        if eps == 0:
            step=0
        elif eps <= 0.8:
            step = 1
        else:
            step = 2
        circuit = trotter_stepper(step, Nqbits, eps, mass, mid).decompose().decompose()
        circuit.measure_all()
        res = do_run(circuit, shots, noise=True, simulator_seed=np.random.randint(1024**2), transpiler_seed=np.random.randint(1024**2))
        ferm_num_trotter_evol_noisy_no_knit = np.append(ferm_num_trotter_evol_noisy_no_knit, fermion_number(res, mid))
        bs_error_trotter_evol_noisy_no_knit = np.append(bs_error_trotter_evol_noisy_no_knit, bootstrap_error(res, mid, shots))
    return {my_seed: {'fermion_number': ferm_num_trotter_evol_noisy_no_knit, 'bootstrap_error': bs_error_trotter_evol_noisy_no_knit}}

In [3]:
noisy_no_knit_trotter_evolution=defaultdict(dict)
for i in range(10):
    print(i)
    noisy_no_knit_trotter_evolution = noisy_no_knit_trotter_evolution | make_runs()

0
0.0
0.2
0.4
0.6
0.8
1.0
1.2
1.4
1.6
1
0.0
0.2
0.4
0.6
0.8
1.0
1.2
1.4
1.6
2
0.0
0.2
0.4
0.6
0.8
1.0
1.2
1.4
1.6
3
0.0
0.2
0.4
0.6
0.8
1.0
1.2
1.4
1.6
4
0.0
0.2
0.4
0.6
0.8
1.0
1.2
1.4
1.6
5
0.0
0.2
0.4
0.6
0.8
1.0
1.2
1.4
1.6
6
0.0
0.2
0.4
0.6
0.8
1.0
1.2
1.4
1.6
7
0.0
0.2
0.4
0.6
0.8
1.0
1.2
1.4
1.6
8
0.0
0.2
0.4
0.6
0.8
1.0
1.2
1.4
1.6
9
0.0
0.2
0.4
0.6
0.8
1.0
1.2
1.4
1.6


In [4]:
with open('data/noisy_no_knit_trotter_evolution.pkl', 'wb') as file:
         pickle.dump(noisy_no_knit_trotter_evolution, file, protocol=pickle.HIGHEST_PROTOCOL)

## resample

In [5]:
with open('data/noisy_no_knit_trotter_evolution.pkl', 'rb') as file:
    noisy_no_knit_trotter_evolution = pickle.load(file)

In [6]:
no_knit_noisy_resampled_fermion_number = np.empty(0)
no_knit_noisy_resampled_bootstrap_error = np.empty(0)
for i in range(9):
    running_resampling = np.empty(0)
    for item in noisy_no_knit_trotter_evolution:
        full_res = noisy_no_knit_trotter_evolution[item]
        ferm_num = full_res['fermion_number'][i]
        bs_error = full_res['bootstrap_error'][i]
        running_resampling = np.append(running_resampling, np.random.normal(ferm_num, bs_error, 1000))
    no_knit_noisy_resampled_fermion_number = np.append(no_knit_noisy_resampled_fermion_number, running_resampling.mean())
    no_knit_noisy_resampled_bootstrap_error = np.append(no_knit_noisy_resampled_bootstrap_error, running_resampling.std())

In [7]:
np.save('data/no_knit_noisy_resampled_fermion_number', no_knit_noisy_resampled_fermion_number)
np.save('data/no_knit_noisy_resampled_bootstrap_error', no_knit_noisy_resampled_bootstrap_error)

# noisy, knitted Trotter evolution

## process data

In [348]:
seeds_noisy_knitted_trotter = []
with open('data/noisy_knitted_trotter_evol/noisy_knitted_seeds_list.txt', 'r') as file:
    for line in file:
        val = line[:-1]
        seeds_noisy_knitted_trotter.append(float(val))
seeds_noisy_knitted_trotter = list(set([int(item) for item in seeds_noisy_knitted_trotter]))

In [349]:
noisy_knitted_trotter_evolution=defaultdict(dict)
for seed in seeds_noisy_knitted_trotter:
    ferm_nums = np.empty(0)
    bootstrap = np.empty(0)
    for eps in np.linspace(0, 1.6, 9):
        eps=np.round(eps, 1)
        print(eps)
        np.random.seed(seed)
        if eps != 0:
            if eps <= 0.8:
                step = 1
            else:
                step = 2
            with open('data/noisy_knitted_trotter_evol/step'+str(step)+\
                  '_epsilon0'+str(eps/step)[-1]+'_count16384_with_noise_knitted_seed'+str(seed)+'.pkl', 'rb') as file:
                res = pickle.load(file)
        else:
            circuit = trotter_stepper(0, Nqbits, eps, mass, mid).decompose().decompose()
            circuit.measure_all()
            np.random.seed(np.random.randint(1024**2))
            res = do_run(circuit, 16384, noise=True, simulator_seed=np.random.randint(1024**2), transpiler_seed=np.random.randint(1024**2)) 
        ferm_nums = np.append(ferm_nums, fermion_number(res, mid))
        bootstrap = np.append(bootstrap, bootstrap_error(res, mid, 16384))
    noisy_knitted_trotter_evolution = noisy_knitted_trotter_evolution | {seed : {'fermion_number': ferm_nums, 'bootstrap_error': bootstrap}}

0.0
0.2
0.4
0.6
0.8
1.0
1.2
1.4
1.6
0.0
0.2
0.4
0.6
0.8
1.0
1.2
1.4
1.6
0.0
0.2
0.4
0.6
0.8
1.0
1.2
1.4
1.6
0.0
0.2
0.4
0.6
0.8
1.0
1.2
1.4
1.6
0.0
0.2
0.4
0.6
0.8
1.0
1.2
1.4
1.6
0.0
0.2
0.4
0.6
0.8
1.0
1.2
1.4
1.6
0.0
0.2
0.4
0.6
0.8
1.0
1.2
1.4
1.6
0.0
0.2
0.4
0.6
0.8
1.0
1.2
1.4
1.6
0.0
0.2
0.4
0.6
0.8
1.0
1.2
1.4
1.6
0.0
0.2
0.4
0.6
0.8
1.0
1.2
1.4
1.6


In [350]:
with open('data/noisy_knitted_trotter_evolution.pkl', 'wb') as file:
         pickle.dump(noisy_knitted_trotter_evolution, file, protocol=pickle.HIGHEST_PROTOCOL)

## resample

In [351]:
with open('data/noisy_knitted_trotter_evolution.pkl', 'rb') as file:
    noisy_knitted_trotter_evolution = pickle.load(file)

In [352]:
knitted_noisy_resampled_fermion_number = np.empty(0)
knitted_noisy_resampled_bootstrap_error = np.empty(0)
for i in range(9):
    running_resampling = np.empty(0)
    for item in seeds_noisy_knitted_trotter:
        full_res = noisy_knitted_trotter_evolution[item]
        ferm_num = full_res['fermion_number'][i]
        bs_error = full_res['bootstrap_error'][i]
        running_resampling = np.append(running_resampling, np.random.normal(ferm_num, bs_error, 1000))
    knitted_noisy_resampled_fermion_number = np.append(knitted_noisy_resampled_fermion_number, running_resampling.mean())
    knitted_noisy_resampled_bootstrap_error = np.append(knitted_noisy_resampled_bootstrap_error, running_resampling.std())

In [353]:
np.save('data/knitted_noisy_resampled_fermion_number', knitted_noisy_resampled_fermion_number)
np.save('data/knitted_noisy_resampled_bootstrap_error', knitted_noisy_resampled_bootstrap_error)